# The Loewner order: logic and a derivation ledger together

This notebook defines the Loewner order $A \preceq B$ of square
matrices and proves its basic properties with
[sympy_matrix_tools](https://github.com/JRF-2018/sympy_matrix_tools).
The last theorems are about the iteration
$V_{k+1} = P^\top V_k P + R$, $V_0 = 0$, which appears in the
Lyapunov and Riccati equations of control theory: if $R$ is
positive semidefinite, the iterates increase,
$V_1 \preceq V_2 \preceq V_3$, and, with induction introduced as an
axiom, $V_k \preceq V_{k+1}$ for all $k$.

Two tools work together:

* **Logic** (`sympy_matrix_tools.fol`): statements are written in the
  logical notation and read by `parse_formula`; a `Theorem` is proved
  backward by tactics (`by_auto`, `by_rule`, `by_rewrite`,
  `by_steps`) or forward by a structured `Proof` (`have` ... `show`).
* **The derivation ledger** (`sympy_matrix_tools.derivation`): the
  algebra that SymPy does not do automatically, such as
  $x^\top (M + N) x = x^\top M x + x^\top N x$, is derived step by
  step, checked symbolically and with random matrices, and then used
  in the logic as a theorem of kind "ledger".

`explain()` shows what each theorem rests on: the axioms (our
definitions), the ledger facts and the theorems used.

The framework aims at plausible arguments whose every step is
checked, not at proofs without gaps from first principles as in Lean.
So we do not hesitate to introduce axioms (definitions, facts about
numbers, induction); we state them openly instead.

In [ ]:
!pip install "sympy>=1.14" git+https://github.com/JRF-2018/sympy_matrix_tools

In [2]:
from sympy import *
from sympy_matrix_tools import *
from sympy_matrix_tools.fol import *
from sympy_matrix_tools.derivation import Derivation, cas, rule

## Symbols

The matrices are $n \times n$ and the vectors $n \times 1$. In SymPy,
a quadratic form $x^\top M x$ is a $1 \times 1$ matrix, so the
"scalars" `a` and `b` of the axioms below are $1 \times 1$ matrices.

The dictionary `S` gives `parse_formula` the objects for the names.
A bound variable whose name is in `S`, such as `M` in `∀M. ...`, is
that `MatrixSymbol`. Include the dimension `n` too, for terms such as
`ZeroMatrix(n, n)`.

In [3]:
n = Symbol("n", integer=True, positive=True)
S = {"n": n}
S.update({k: MatrixSymbol(k, n, n) for k in "A B C M N P R".split()})
S.update({k: MatrixSymbol(k, n, 1) for k in ["x", "y"]})
S.update({k: MatrixSymbol(k, 1, 1) for k in ["a", "b"]})
A, B, C, M, N, P, R, x = [S[k] for k in "A B C M N P R x".split()]

## Axioms: the definitions

`psd(M)` (positive semidefinite) and `mat_le(A, B)` ($A \preceq B$)
are defined here, in the notebook; they are not part of the library.
`nonneg(a)` means $a \geq 0$, and two facts about it are assumed. The
definitions are written with `↔`; a tactic uses such a theorem in
both directions.

In [4]:
nonneg_add = Theorem("nonneg_add",
                     "∀a b. nonneg(a) ∧ nonneg(b) → nonneg(a + b)",
                     axiom=True, symbols=S)
nonneg_zero = Theorem("nonneg_zero", "nonneg(ZeroMatrix(1, 1))",
                      axiom=True, symbols=S)
psd_def = Theorem("psd_def", "∀M. psd(M) ↔ (∀x. nonneg(x.T*M*x))",
                  axiom=True, symbols=S)
le_def = Theorem("le_def", "∀A B. mat_le(A, B) ↔ psd(B - A)",
                 axiom=True, symbols=S)
for th in [nonneg_add, nonneg_zero, psd_def, le_def]:
    print(th)

axiom nonneg_add: ∀a b. (nonneg(a) ∧ nonneg(b)) → nonneg(a + b)
axiom nonneg_zero: nonneg(0)
axiom psd_def: ∀M. psd(M) ↔ (∀x. nonneg(x.T*M*x))
axiom le_def: ∀A B. mat_le(A, B) ↔ psd(-A + B)


A statement is an ordinary SymPy formula. `formula_str` prints it in
the logical notation, and `parse_formula` reads the printed text
back.

In [5]:
f = parse_formula("∀M. psd(M) ↔ (∀x. nonneg(x.T*M*x))", symbols=S)
print(f)
print(formula_str(f))
parse_formula(formula_str(f), symbols=S) == f

Q.ForAll(Lambda(M, Equivalent(PredApply(psd, M), Q.ForAll(Lambda(x, PredApply(nonneg, x.T*M*x))))))
∀M. psd(M) ↔ (∀x. nonneg(x.T*M*x))


True

## The first ledger: quadratic forms

The logic does not know algebra. SymPy evaluates some algebra
automatically (`(P*x).T` is `x.T*P.T`, `x.T*0*x` is `0`, and
`(C - B) + (B - A)` is `C - A`), but it does not distribute a
product over a sum. These identities are derived in a ledger. Each
step is justified by `cas(...)` (a computation by SymPy), and
`check()` compares both sides with random rational matrices.

In [6]:
d1 = Derivation("Quadratic forms")
d1.calc("quad_add", x.T*(M + N)*x,
        (x.T*M*x + x.T*N*x, cas(mat_expand)))
d1.calc("congr_diff", P.T*(B - A)*P,
        (P.T*B*P - P.T*A*P, cas(mat_expand)))
print(d1.check())
print(d1.report())

True
Derivation: Quadratic forms
[derived] quad_add: x.T*(M + N)*x = x.T*M*x + x.T*N*x
  x.T*(M + N)*x
  = x.T*M*x + x.T*N*x    by cas mat_expand    symbolic: ok    numeric: ok 3/3
[derived] congr_diff: P.T*(-A + B)*P = -P.T*A*P + P.T*B*P
  P.T*(-A + B)*P
  = -P.T*A*P + P.T*B*P    by cas mat_expand    symbolic: ok    numeric: ok 3/3
Summary: 2 steps, 0 asserted, 0 symbolic failures, 0 numeric failures, 0 open obligations.


`theorem(name)` turns a derived fact into a `Theorem` of kind
"ledger", universally quantified over its symbols (the hypotheses of
the ledger, if any, become premises). `verify()` requires that every
step is justified and checked symbolically, and that no numerical
check fails.

In [7]:
quad_add = d1.theorem("quad_add")
congr_diff = d1.theorem("congr_diff")
print(quad_add.verify(), congr_diff.verify())
print(quad_add.explain())
print(congr_diff)

True True
ledger quad_add: ∀M N x. x.T*(M + N)*x = x.T*M*x + x.T*N*x
  by Derivation "Quadratic forms": 1 step checked symbolically and numerically (3 trials)
  verified: yes
ledger congr_diff: ∀A B P. P.T*(-A + B)*P = -P.T*A*P + P.T*B*P


## Backward proofs

### psd_unfold

The definition `psd_def` in the form of an elimination rule. `by_auto`
searches a proof backward with the given theorems.

In [8]:
psd_unfold = Theorem("psd_unfold", "∀M x. psd(M) → nonneg(x.T*M*x)",
                     by_auto(psd_def), symbols=S)
psd_unfold.verify()

True

### psd_add, step by step

$M, N \succeq 0 \Rightarrow M + N \succeq 0$. The proof unfolds the
definition backward (`by_rule`), rewrites the quadratic form by the
ledger fact `quad_add` (`by_rewrite`), and lets `by_auto` finish. We
first run the steps by hand to see the proofstates: each `prem` is a
subgoal, and `gl*` is the goal.

In [9]:
z = make_proofstate(Theorem("t", "∀M N. psd(M) ∧ psd(N) → psd(M + N)",
                            symbols=S).statement)
print_proofstate(z, pretty=True)
z = by_rule(psd_def)(z)
print_proofstate(z, pretty=True)
z = by_rewrite(quad_add)(z)
print_proofstate(z, pretty=True)
z = by_auto(nonneg_add, psd_unfold)(z)
print_proofstate(z, pretty=True)

prem 0: ∀M N. (psd(M) ∧ psd(N)) → psd(M + N)
gl*: ∀M1 N1. (psd(M1) ∧ psd(N1)) → psd(M1 + N1)
prem 0: ∀M N x. (psd(M) ∧ psd(N)) → nonneg(x.T*(M + N)*x)
gl*: ∀M1 N1. (psd(M1) ∧ psd(N1)) → psd(M1 + N1)
prem 0: ∀M N x. (psd(M) ∧ psd(N)) → nonneg(x.T*M*x + x.T*N*x)
gl*: ∀M1 N1. (psd(M1) ∧ psd(N1)) → psd(M1 + N1)


gl*: ∀M N. (psd(M) ∧ psd(N)) → psd(M + N)


The same steps as a proof of a theorem:

In [10]:
psd_add = Theorem("psd_add", "∀M N. psd(M) ∧ psd(N) → psd(M + N)",
                  by_steps(by_rule(psd_def), by_rewrite(quad_add),
                           by_auto(nonneg_add, psd_unfold)),
                  symbols=S)
psd_add.verify()
print(psd_add.explain())

theorem psd_add: ∀M N. (psd(M) ∧ psd(N)) → psd(M + N)
  proof: rule(psd_def); rewrite(quad_add); auto(nonneg_add, psd_unfold)
  axioms: psd_def, nonneg_add
  ledger facts: quad_add
  theorems: psd_unfold
  verified: yes


### psd_congr: an instance

$M \succeq 0 \Rightarrow P^\top M P \succeq 0$. Unfolding the goal
gives `nonneg(x.T*P.T*M*P*x)`, which is `psd_unfold` at the vector
$P x$. Unification cannot find $P x$ in `x.T*P.T*M*P*x`, so we give
the instance explicitly: `instance(x=P*x)` replaces the bound `x`
and binds the new free symbols again.

In [11]:
print(psd_unfold.instance(x=P*x))
psd_congr = Theorem("psd_congr", "∀M P. psd(M) → psd(P.T*M*P)",
                    by_auto(psd_def, psd_unfold.instance(x=P*x)),
                    symbols=S)
psd_congr.verify()

instance psd_unfold[x := P*x]: ∀M P x. psd(M) → nonneg(x.T*P.T*M*P*x)


True

### Reflexivity and transitivity

SymPy's automatic evaluation does the small algebra: `x.T*0*x` is `0`
and `A - A` is `0`. For transitivity, the instance of `psd_add` at
$M = C - B$, $N = B - A$ concludes `psd(C - A)`, because SymPy
simplifies $(C - B) + (B - A)$.

In [12]:
psd_zero = Theorem("psd_zero", "psd(ZeroMatrix(n, n))",
                   by_auto(psd_def, nonneg_zero), symbols=S)
le_refl = Theorem("le_refl", "∀A. mat_le(A, A)",
                  by_auto(le_def, psd_zero), symbols=S)
print(psd_add.instance(M=C - B, N=B - A))
le_trans = Theorem("le_trans",
                   "∀A B C. mat_le(A, B) ∧ mat_le(B, C) → mat_le(A, C)",
                   by_auto(le_def, psd_add.instance(M=C - B, N=B - A)),
                   symbols=S)
print(le_refl.verify(), le_trans.verify())
print(le_trans.explain())

instance psd_add[M := -B + C, N := -A + B]: ∀A B C. (psd(-A + B) ∧ psd(-B + C)) → psd(-A + C)


True True
theorem le_trans: ∀A B C. (mat_le(A, B) ∧ mat_le(B, C)) → mat_le(A, C)
  proof: auto(le_def, psd_add[M := -B + C, N := -A + B])
  axioms: le_def, psd_def, nonneg_add
  ledger facts: quad_add
  theorems: psd_unfold, psd_add
  verified: yes


## Forward proofs

### Addition

$A \preceq B \wedge C \preceq D \Rightarrow A + C \preceq B + D$.

A `Proof` is written forward, like Isar of Isabelle: the variables
of the statement are fixed, its assumptions are named `h1`, `h2`, ...,
and each `have` derives a new fact from the facts named in `using`.
Each step is proved at once (a wrong step raises `ValueError` there).
`show` derives the conclusion and returns the theorem, whose proof
combines the steps.

In `p3`, SymPy writes $(B - A) + (D - C)$ as `-A + B - C + D`, and
the unifier finds $M = B - A$, $N = D - C$ of `psd_add` in it.

In [13]:
D = MatrixSymbol("D", n, n)
S["D"] = D
pf0 = Proof("le_add",
            "∀A B C D. mat_le(A, B) ∧ mat_le(C, D) → mat_le(A + C, B + D)",
            symbols=S)
pf0.have("p1", "psd(B - A)", by_auto(le_def), using=["h1"])
pf0.have("p2", "psd(D - C)", by_auto(le_def), using=["h2"])
pf0.have("p3", "psd((B - A) + (D - C))", by_auto(psd_add),
         using=["p1", "p2"])
le_add = pf0.show(by_auto(le_def), using=["p3"])
print(pf0)

proof le_add: ∀A B C D. (mat_le(A, B) ∧ mat_le(C, D)) → mat_le(A + C, B + D)
  fix A B C D
  assume h1: mat_le(A, B)
  assume h2: mat_le(C, D)
  have p1: psd(-A + B)
      using h1  by auto(le_def)
  have p2: psd(-C + D)
      using h2  by auto(le_def)
  have p3: psd(-A + B - C + D)
      using p1, p2  by auto(psd_add)
  show mat_le(A + C, B + D)
      using p3  by auto(le_def)
qed


### Monotonicity of one step

$A \preceq B \Rightarrow P^\top A P + R \preceq P^\top B P + R$.

The step `f` uses the ledger fact `congr_diff` from right to left.

In [14]:
pf = Proof("lyap_mono",
           "∀A B P R. mat_le(A, B) → mat_le(P.T*A*P + R, P.T*B*P + R)",
           symbols=S)
pf.have("d", "psd(B - A)", by_auto(le_def), using=["h1"])
pf.have("e", "psd(P.T*(B - A)*P)", by_auto(psd_congr), using=["d"])
pf.have("f", "psd(P.T*B*P - P.T*A*P)",
        by_steps(by_rewrite(congr_diff, reverse=True), by_auto()),
        using=["e"])
lyap_mono = pf.show(by_auto(le_def), using=["f"])
print(pf)

proof lyap_mono: ∀A B P R. mat_le(A, B) → mat_le(R + P.T*A*P, R + P.T*B*P)
  fix A B P R
  assume h1: mat_le(A, B)
  have d: psd(-A + B)
      using h1  by auto(le_def)
  have e: psd(P.T*(-A + B)*P)
      using d  by auto(psd_congr)
  have f: psd(-P.T*A*P + P.T*B*P)
      using e  by rewrite_rev(congr_diff); auto()
  show mat_le(R + P.T*A*P, R + P.T*B*P)
      using f  by auto(le_def)
qed


In [15]:
print(lyap_mono.explain())

theorem lyap_mono: ∀A B P R. mat_le(A, B) → mat_le(R + P.T*A*P, R + P.T*B*P)
  proof: structured proof (4 steps)
  axioms: le_def, psd_def
  ledger facts: congr_diff
  theorems: psd_unfold, psd_congr
  verified: yes


## The second ledger: the iterates

The iterates are $V_1 = R$, $V_2 = P^\top R P + R$ and
$V_3 = P^\top V_2 P + R$. The ledger expands $V_3$.

In [16]:
V2 = P.T*R*P + R
V3 = P.T*V2*P + R
d2 = Derivation("Lyapunov iterates")
d2.calc("V3_expand", V3, (P.T*P.T*R*P*P + P.T*R*P + R, cas(mat_expand)))
print(d2.check())
print(d2.report())
V3_expand = d2.theorem("V3_expand")

True
Derivation: Lyapunov iterates
[derived] V3_expand: R + P.T*(R + P.T*R*P)*P = P.T**2*R*P**2 + R + P.T*R*P
  R + P.T*(R + P.T*R*P)*P
  = P.T**2*R*P**2 + R + P.T*R*P    by cas mat_expand    symbolic: ok    numeric: ok 3/3
Summary: 1 step, 0 asserted, 0 symbolic failures, 0 numeric failures, 0 open obligations.


## The iterates increase

$R \succeq 0 \Rightarrow V_1 \preceq V_2 \wedge V_2 \preceq V_3$.

Here $V_1 \preceq V_2$ is `lyap_mono` at $A = 0$, $B = R$ (with
$V_0 = 0 \preceq R = V_1$), and $V_2 \preceq V_3$ is `lyap_mono` at
$A = V_1$, $B = V_2$, which `by_auto` finds by unification. The step
`v23e` states $V_3$ in the expanded form of the ledger.

In [17]:
Z = ZeroMatrix(n, n)
print(lyap_mono.instance(A=Z, B=R))
pf2 = Proof("iterates_mono",
            "∀P R. psd(R) → mat_le(R, R + P.T*R*P)"
            " ∧ mat_le(R + P.T*R*P, R + P.T*R*P + P.T*P.T*R*P*P)",
            symbols=S)
pf2.have("v01", "mat_le(ZeroMatrix(n, n), R)", by_auto(le_def),
         using=["h1"])
pf2.have("v12", "mat_le(R, R + P.T*R*P)",
         by_auto(lyap_mono.instance(A=Z, B=R)), using=["v01"])
pf2.have("v23", "mat_le(R + P.T*R*P, R + P.T*(R + P.T*R*P)*P)",
         by_auto(lyap_mono), using=["v12"])
pf2.have("v23e", "mat_le(R + P.T*R*P, R + P.T*R*P + P.T*P.T*R*P*P)",
         by_steps(by_rewrite(V3_expand, reverse=True), by_auto()),
         using=["v23"])
iterates_mono = pf2.show(by_auto(), using=["v12", "v23e"])
print(pf2)

instance lyap_mono[A := 0, B := R]: ∀P R. mat_le(0, R) → mat_le(R, R + P.T*R*P)


proof iterates_mono: ∀P R. psd(R) → (mat_le(R, R + P.T*R*P) ∧ mat_le(R + P.T*R*P, P.T**2*R*P**2 + R + P.T*R*P))
  fix P R
  assume h1: psd(R)
  have v01: mat_le(0, R)
      using h1  by auto(le_def)
  have v12: mat_le(R, R + P.T*R*P)
      using v01  by auto(lyap_mono[A := 0, B := R])
  have v23: mat_le(R + P.T*R*P, R + P.T*(R + P.T*R*P)*P)
      using v12  by auto(lyap_mono)
  have v23e: mat_le(R + P.T*R*P, P.T**2*R*P**2 + R + P.T*R*P)
      using v23  by rewrite_rev(V3_expand); auto()
  show mat_le(R, R + P.T*R*P) ∧ mat_le(R + P.T*R*P, P.T**2*R*P**2 + R + P.T*R*P)
      using v12, v23e  by auto()
qed


Finally $V_1 \preceq V_3$ by transitivity, in a short forward proof
that uses the previous theorem.

In [18]:
pf3 = Proof("iterates_trans",
            "∀P R. psd(R) → mat_le(R, R + P.T*R*P + P.T*P.T*R*P*P)",
            symbols=S)
pf3.have("both", "mat_le(R, R + P.T*R*P)"
         " ∧ mat_le(R + P.T*R*P, R + P.T*R*P + P.T*P.T*R*P*P)",
         by_auto(iterates_mono))
iterates_trans = pf3.show(by_auto(le_trans.instance(B=R + P.T*R*P)),
                          using=["both"])
print(iterates_trans.explain())

theorem iterates_trans: ∀P R. psd(R) → mat_le(R, P.T**2*R*P**2 + R + P.T*R*P)
  proof: structured proof (2 steps)
  axioms: le_def, psd_def, nonneg_add
  ledger facts: congr_diff, V3_expand, quad_add
  theorems: psd_unfold, psd_congr, lyap_mono, iterates_mono, psd_add, le_trans
  verified: yes


This theorem rests on the two definitions, the axiom `nonneg_add` and
three ledger facts. The ledger facts are trusted to SymPy's
computation (and are additionally checked with random matrices);
everything else is a proof in the logic.

## For all k: induction as an axiom

So far only $V_1 \preceq V_2 \preceq V_3$ is proved. To prove
$V_k \preceq V_{k+1}$ for all $k$, we need induction, which the
logic does not have. We simply **introduce it as an axiom**.

This is in the spirit of the framework. It does not aim at a proof
without gaps from first principles, as Lean does; it aims at a
plausible argument whose every step is checked. Axioms are fine
there, as long as they are plausible and stated openly: `explain()`
lists every axiom that a theorem rests on. (What is not fine is to
state the claim itself, or a step of it, as an axiom.)

The iterates are a matrix-valued function `V(k)` of a natural
number `k` (a `MatrixFunction`), defined by two axioms. `P` and `R`
are fixed matrices here.

In [19]:
k = Symbol("k", integer=True, nonnegative=True)
V = MatrixFunction("V", n, n)
S.update(k=k, V=V)
nat_induct = Theorem("nat_induct",
                     "∀Phi k. Phi(0) ∧ (∀k. Phi(k) → Phi(k + 1)) → Phi(k)",
                     axiom=True, symbols=S)
V_0 = Theorem("V_0", "V(0) = ZeroMatrix(n, n)", axiom=True, symbols=S)
V_step = Theorem("V_step", "∀k. V(k + 1) = P.T*V(k)*P + R",
                 axiom=True, symbols=S)
for th in [nat_induct, V_0, V_step]:
    print(th)

axiom nat_induct: ∀Phi k. (Phi(0) ∧ (∀k1. Phi(k1) → Phi(k1 + 1))) → Phi(k)
axiom V_0: V(0) = 0
axiom V_step: ∀k. V(k + 1) = R + P.T*V(k)*P


`by_induct(nat_induct, "k")` is the induction on `k`, like `induct`
of Isabelle: it instantiates `Phi` with $\lambda k.$ (the goal), and
leaves the base case and the step as subgoals. We run the steps by
hand first.

In [20]:
z = make_proofstate(Theorem("t", "∀k. psd(R) → mat_le(V(k), V(k + 1))",
                            symbols=S).statement)
z = by_induct(nat_induct, "k")(z)
print_proofstate(z, pretty=True)

prem 0: psd(R) → mat_le(V(0), V(1))
prem 1: ∀k. (psd(R) ∧ mat_le(V(k), V(k + 1))) → mat_le(V(k + 1), V(k + 2))
gl*: ∀k1. psd(R) → mat_le(V(k1), V(k1 + 1))


The base case: by the instance of `V_step` at $k = 0$ and `V_0`,
$V_1 = R$, and $0 \preceq R$ is `psd(R)` by `le_def`.

In [21]:
z = by_steps(by_rewrite(V_step.instance(k=0)), by_rewrite(V_0),
             by_rule(le_def))(z)
print_proofstate(z, pretty=True)

prem 0: ∀k. (psd(R) ∧ mat_le(V(k), V(k + 1))) → mat_le(V(k + 1), V(k + 2))
gl*: ∀k1. psd(R) → mat_le(V(k1), V(k1 + 1))


The step: rewrite $V_{k+2}$ by the instance of `V_step` at $k + 1$,
then only the first occurrence of $V_{k+1}$ by `V_step`
(`occurrence=0`). What remains is `lyap_mono` applied to the
induction hypothesis $V_k \preceq V_{k+1}$.

In [22]:
z = by_steps(by_rewrite(V_step.instance(k=k + 1)),
             by_rewrite(V_step, occurrence=0))(z)
print_proofstate(z, pretty=True)
z = by_auto(lyap_mono)(z)
print_proofstate(z, pretty=True)

prem 0: ∀k. (psd(R) ∧ mat_le(V(k), V(k + 1))) → mat_le(R + P.T*V(k)*P, R + P.T*V(k + 1)*P)
gl*: ∀k1. psd(R) → mat_le(V(k1), V(k1 + 1))
gl*: ∀k. psd(R) → mat_le(V(k), V(k + 1))


The same steps as a theorem:

In [23]:
V_mono = Theorem("V_mono", "∀k. psd(R) → mat_le(V(k), V(k + 1))",
                 by_steps(by_induct(nat_induct, "k"),
                          by_rewrite(V_step.instance(k=0)),
                          by_rewrite(V_0), by_rule(le_def),
                          by_rewrite(V_step.instance(k=k + 1)),
                          by_rewrite(V_step, occurrence=0),
                          by_auto(lyap_mono)),
                 symbols=S)
V_mono.verify()
print(V_mono.explain())

theorem V_mono: ∀k. psd(R) → mat_le(V(k), V(k + 1))
  proof: induct(nat_induct, k); rewrite(V_step[k := 0]); rewrite(V_0); rule(le_def); rewrite(V_step[k := k + 1]); rewrite(V_step, occurrence=0); auto(lyap_mono)
  axioms: nat_induct, V_step, V_0, le_def, psd_def
  ledger facts: congr_diff
  theorems: psd_unfold, psd_congr, lyap_mono
  verified: yes


A second induction uses the first theorem: every iterate is positive
semidefinite, $0 \preceq V_k$ (the step is by transitivity with
`V_mono`).

In [24]:
V_nonneg = Theorem("V_nonneg", "∀k. psd(R) → mat_le(ZeroMatrix(n, n), V(k))",
                   by_steps(by_induct(nat_induct, "k"), by_rewrite(V_0),
                            by_auto(le_refl, le_trans, V_mono)),
                   symbols=S)
V_nonneg.verify()
print(V_nonneg.explain())

theorem V_nonneg: ∀k. psd(R) → mat_le(0, V(k))
  proof: induct(nat_induct, k); rewrite(V_0); auto(le_refl, le_trans, V_mono)
  axioms: nat_induct, V_0, le_def, psd_def, nonneg_zero, nonneg_add, V_step
  ledger facts: quad_add, congr_diff
  theorems: psd_zero, le_refl, psd_unfold, psd_add, le_trans, psd_congr, lyap_mono, V_mono
  verified: yes


The axioms listed are our definitions (`psd_def`, `le_def`, `V_0`,
`V_step`), facts about nonnegative numbers (`nonneg_zero`,
`nonneg_add`) and induction (`nat_induct`); the ledger facts are the
algebra. A reader who accepts these accepts the theorem.

## What is rejected

A non-theorem is not proved: the search fails and `verify()` raises
`ValueError`.

In [25]:
try:
    Theorem("le_sym", "∀A B. mat_le(A, B) → mat_le(B, A)",
            by_auto(le_def, psd_add, psd_congr, max_nodes=300),
            symbols=S).verify()
except ValueError as e:
    print(e)

le_sym: the proof failed.


A wrong step of a forward proof fails where it is written:
$A \preceq B$ does not give $P^\top A P \succeq 0$.

In [26]:
pf4 = Proof("wrong", "∀A B P. mat_le(A, B) → psd(P.T*A*P)", symbols=S)
try:
    pf4.have("d", "psd(A)", by_auto(le_def, max_nodes=100), using=["h1"])
except ValueError as e:
    print(e)

wrong.d: the proof failed.


A wrong identity in a ledger is found by the numerical check, even
if it claims to be a computation by SymPy, and it cannot become a
theorem.

In [27]:
d3 = Derivation("A wrong identity")
d3.calc("congr_bad", P.T*(B - A)*P,
        (P.T*B*P - P*A*P.T, cas(mat_expand)))
print(d3.check())
print(d3.report())
try:
    d3.theorem("congr_bad").verify()
except ValueError as e:
    print(e)

False
Derivation: A wrong identity
[derived] congr_bad: P.T*(-A + B)*P = -P*A*P.T + P.T*B*P
  P.T*(-A + B)*P
  = -P*A*P.T + P.T*B*P    by cas mat_expand    symbolic: FAILED    numeric: FAILED 0/3
Summary: 1 step, 0 asserted, 1 symbolic failure, 1 numeric failure, 0 open obligations.
congr_bad: step 1 of congr_bad failed the numerical check (see its counterexample).
